## Acronyms
- **APC**: Anterior Piriform Cortex
- **OE**: Olfactory Epithelium
- **OSN**: Olfactory Sensory Neuron
- **OR**: Olfactory Receptor
- **OB**: Olfactory Bulb
- **LOT**: Lateral Olfactory Tract
- **MTC**: Mitral/Tufted Cell
- **PN**: Pyramidal Neuron
- **PV-IN**: Parvalbumin Interneuron
- **SST-IN**: Somatostatin Interneuron
- **VIP-IN**: Vasoactive Intestinal Peptide Interneuron
- **LTP**: Long Term Potentiation


## Contact

In case of questions or if you run into problems contact Sander de Haan at sdehaan@ethz.ch

# Learning in the olfactory cortex


In this tutorial, we will build a neural network inspired by the structure of the anterior piriform cortex (APC), a brain region responsible for encoding odor identity in mammals.<br> 
This tutorial is divided into three sections: The first section introduces the biological basis of odor identity encoding, the second section describes how we translate this into a computational model, and the last section consists of the actual simulation part. For each section, you will be asked to answer the questions at the end (one or two sentences are enough). For the simulation section, you will also need to complete the code in some of the functions to make the simulation run.


## 1. Biology

### 1.1 Smell
The smell of freshly brewed coffee is the result of various odorants activating olfactory sensory neurons (OSNs) by binding to olfactory receptors (ORs) in the nasal olfactory epithelium (OE) (Figure 1 $\textbf{A}$). Each OSN expresses one specific OR-subtype which is tuned to particular physicochemical properties of molecules.  
Axons of OSNs expressing identical OR-subtypes converge onto the same glomerulus in the olfactory bulb (OB), creating a highly ordered input structure.


<figure>
    <center>
        <img src="Figures\IFig1.png"/>
        <figcaption align = "justify"> <b>Figure 1: Mouse olfactory system and odor identity encoding in the APC: A) </b>  Odorants bind to ORs on OSNs. Axons of OSNs expressing identical OR subtypes converge onto the same glomerulus. In the OB, signal preprocessing occurs before MTCs relay the input to PNs in the APC. Inhibitory neurons are shown in red. Adapted from <a href="#ref">Shepherd (1991)</a> with inspiration from <a href="#ref">Uchida et al. (2014)</a>. <b> B1) </b> Afferent input via L1a (axons of MTCs) activates a subset of PNs (triangles), while many neurons either remain below their threshold level or are not activated at all. <b> B2) </b> Active PNs recruit additional cells into the neuronal ensemble through recurrent intracortical connections, but only if these neurons have already received input via afferents. Recurrent input alone is not strong enough to recruit inactive neurons. </figcaption>    
    </center>
</figure>



### 1.2 Anterior piriform Cortex \& odor identity encoding
The piriform cortex, a three-layered paleocortex, receives direct input from the OB via axons of Mitral and Tuftet cells (MTCs). Interestingly, the topographical order of the input, meticulously maintained throughout the OB, is lost as the axons of the MTCs project diffusely to the APC.

Pyramidal neurons (PNs) in the APC receive afferent input from MTCs at the distal part of their apical dendrite (Figure 1 $\textbf{B1}$). To consistently activate a PN, it takes 5-10 co-active MTCs.
As a result, only a tiny subset of PNs fire, while the remaining cells may experience depolarization below their activation threshold. The active subset of neurons recruits additional PNs into the ensemble through recurrent intracortical connections, targeting the PNs dendrites closer to the soma (Figure 1 $\textbf{B2}$). The recurrent activation is strong enough to activate the previously depolarized neurons but too weak to recruit PNs that did not receive any signal from the OB at all. As network activity intensifies, recurrent inhibition is triggered, suppressing the response of neurons recruited later in the process. Consequently, odor identity is encoded by unique ensembles of active neurons in a concentration-invariant manner within the APC.

The strengthening of recurrent intracortical synapses between PNs is hypothesized to be the driving force behind the formation of long-lasting ensemble odor representations.

### 1.3 Disinhibitory circuit

With their synapses close to the soma of PNs, parvalbumin interneurons (PV-INs) exert rapid and potent inhibition and are thought to be critical in establishing the sparse and distributed activity patterns within PN ensembles.
The inhibitory synapses of somatostatin-positive interneurons (SST-INs) on the proximal apical dendrite of PNs, however, are optimally located to regulate intracortical synaptic plasticity. [Canto-Bustos et al. (2022)](#ref) showed that long-term potentiation (LTP) in those PN-to-PN synapses can be gated by inactivating SST-INs and that vasoactive intestinal polypeptide (VIP) expressing cells are responsible for strong inhibition of those SST-INs.
A teaching signal that activates this disinhibitory VIP-to-SST-to-PN circuit could shape neuronal ensemble formation and may be recruited on novel, salient, or emotionally relevant odors.

<figure>
    <center>
        <img src="Figures\IFig3.png"/>
        <figcaption align = "center"> <b>Figure 2: Simplified APC microcircuit: </b> Odor input arrives diffusely at the apical dendrites of PNs. PNs recurrently project to other PNs uniformly throughout the APC. Connections to inhibitory SST- and PV-INs, lead to feedback inhibition. A teaching signal can activate the disinhibitory VIP-SST-PN circuit and open a window for LTP in intracortical PN-to-PN connections. </figcaption>    
    </center>
</figure>


Based on this model, we will develop a simplified version of the APC microcircuit (Figure 2) and explore the effect of a teaching signal that activates the disinhibitory VIP-to-SST circuit during odor learning.


**Questions**
1. Can different OR subtypes show affinity to the same molecule?
2. Do you expect that without a teaching signal, odor identity is not learned at all? Name a learning algorithm that does not require a teaching signal.
3. Odor intensity is encoded by the same neuronal ensemble that encodes odor identity. Since the membership code is invariant to concentration, how else could it be achieved?

**Your Answers**
1. ...
2. ...
3. ...

## 2. Model

### 2.1 Network
<a id='eq'></a>
We will use a rate-based model, which means that we will not model the spiking of the neurons themselves. Rather, we will model the membrane potential of the neurons using first-order differential equations and then use a non-linear activation function to convert it into instantaneous activity.
Our network will consist of 50 Pyramidal neurons (PN's), 8 SST neurons, 4 PV neurons and 1 VIP neuron.

The following differential equations describe the membrane potential of our neurons:


\begin{align}
\frac{d\vec{V}_{PN}}{dt} ={} & W_{Input}^{PN} \cdot \vec{x}(t) + W_{PN}^{PN} \cdot \vec{r}_{PN}(t) -k_{PN} \vec{V}_{PN}(t)\\ & - \min\left(W_{Input}^{PN} \cdot \vec{x}(t),W_{SST}^{PN} \cdot \vec{r}_{SST}(t)\right) - W_{PV}^{PN} \cdot \vec{r}_{PV}(t) \\[1em]
\frac{d\vec{V}_{PV}}{dt} ={} & W_{PN}^{PV} \cdot \vec{r}_{PN}(t)-k_{PV} \vec{V}_{PV}(t)\\[1em]
\frac{d\vec{V}_{SST}}{dt} ={} & W_{PN}^{SST} \cdot \vec{r}_{PN}(t) -k_{SST} \vec{V}_{SST}(t) - W_{VIP}^{SST} \cdot \vec{r}_{VIP}(t) \\[1em]
\frac{d\vec{V}_{VIP}}{dt} ={} & \vec{\tau}(t) -k_{VIP} \vec{V}_{VIP}(t) \\[1em]
\vec{r}_X(t) ={} & ReLU(\vec{V_X}(t)) = \max(0,\min((\delta_X \cdot \vec{V_X}(t) - \theta_X), r_X^{max}))
\end{align}


<br>
<br>


| Parameter     | Value         | Name in Code  | Explanation   |
| :------------ | :------------ | :------------ | :------------ |
|  $\vec{V}_{X}$ |  $\mathbb{R}$ | V\_{X} | Membrane potential of X |
|  $r_{X}(t)$ | $\mathbb{R}_+^{n}$ | Activity\_X | Activity (rate) of X |
|  $r_X^{max}$ | $\mathbb{R}_+$ | Rmax\_X | Maximal activity (rate) of X |
|  $k_{X}$ | const.  | k\_X | EPSP decay rate for X|
|  $W_{X}^{Y}$ | $\mathbb{R}_+^{n \times m}$ | weightX\_Y |  Weight matrix for $X\to Y$ connection <br> with $n = \#Y$ and $m = \#X $ <br> only $W_{PN}^{PN} $ is time dependent |
|  $\vec{x}(t)$ | $\mathbb{N}_+^n$ | input | Input at time $t$ |
|  $\tau(t)$ |  const. | teacher | Magnitude of teacher input |
|  $\bar{r}(t)$ | $\mathbb{R}_+$ |  r\_avrg | Average rate from $T-t$ to $t$ |
|  T | 100 $ms$ | timeAvrg | Time-window of integration  <be> for weight update |
|  $\alpha$ | 0.8 | alpha | Rate of regularization in OJA |
|  $\lambda$ | $1.2 \times 10^{-6}$ | learning\_rate | Learning rate |
|  $\theta$ | $\mathbb{R}_+$ | Thresh\_X | Activity threshold |
|  $\delta$ | $\mathbb{R}_+$ | Gain\_X | Activity gain |
|  $dt$ | 1 ms | -- | Time increment |

$$X,Y \in \{PN,SST,PV,VIP\}$$



We model the differential equations using the [forward Euler](https://en.wikipedia.org/wiki/Euler_method) method with a time-step of $dt = 1~ms$.



### 2.2 Odor
We model odor as binary vectors. A $"1"$ indicates the presence of a certain odorant in the odor, and a $"0"$ indicates its absence. For simplicity, we restrict the size of an odor to $10$ different odorants and define that an odor always consists of $2$ odorants.
One of the odors will be linked to a teaching signal and one odor will function as a control. These two odors will be presented in an alternating fashion to our neuronal network. Further, random odors are presented to the network to test the noise resistance of the network (Figure 3).


<figure>
    <center>
        <img src="Figures\MFig2.png"/>
        <figcaption align = "justify"> <b>Figure 3: Odor distribution.</b> Blue odors are linked to a teaching signal and darkgray inputs serve as controls. Additional random odors are presented during training to evaluate learning with noisy input.
 </figcaption>    
    </center>
</figure>




**Questions**
1. How many distinct odors does the odor distribution contain?
2. According to [Equation 1](#eq), the magnitude of SST inhibition cannot surpass which portion of the input to PNs?
3. Instead of modeling inhibition by subtracting the effect of inhibitory neurons, how else can inhibition be modeled?

**Your Answers:**
1. ...
2. ...
3. ...

## 3. Simulation

Before we start the simulation, make sure you installed all the packages from the `requirements.txt` file.

If you use conda:
```
conda create --name odorexo python=3.10
conda activate odorexo
where pip
<Path to pip.exe within the odorexo environment> install -r requirements.txt
```

If you use pip without any virtual enviroment:
```
pip install -r requirements.txt
```

Now let's start.
Here we create a class that we call `APC_circuit`. For now, this class contains the `__init__` constructor, which is used to initialize the model parameters (e.g number of neurons, connection probabilities, maximal firing rates etc.). Further, the class contains a function to initialize the network connections and to generate the odor inputs as well as some plotting functions to later visualize the results. **You do not need to understand the details and can just run the cell.**

In [1]:
# Import packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import time
import networkx as nx
import pandas as pd
import seaborn as sns
from netgraph import Graph
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LinearSegmentedColormap
from plottingFunctions import *
import collections


class APC_circuit:
    def __init__(self, time_ms = 40000, learning_rate = 0.000001, seed = 1):
        ### Store parameters passed by the function ###

        # Define the number of PN, SST & VIP neurons in the network
        self.n_PN = 50
        self.n_PV = 4
        self.n_SST = 8
        self.n_VIP = 1

        # Size of input vector (number of distinct olfactory receptors)
        self.n_inputs = 10

        # Number of iterations (dt = 1 ms)
        self.time_ms = time_ms

        # Learning rate during weight update in Oja's rule
        self.learning_rate = learning_rate

        # Seed used for calls to numpy.random
        self.seed = seed
        np.random.seed(self.seed)

        # Set style for all ploting functions.
        plt.style.use('seaborn-v0_8-paper')
        self.color = ['#1f77b4','#525252']

        ###############################################
        ### Set parameters for neurons and the network ###


        self.n_randInputs = 1
        self.con_n_PV_to_PN = 1 				# Number of PV neurons that are connected to each PN at random. 23%

        # Conectivity parameters
        self.numberOfActiveInputs = 2 			# Number of active inputs in the inputarray (Active Receptors / Glomeruli for a given odour so to speak).
        self.numberOfGlomeruliToPN = 2 			# Number of Glomeruli each PN is connected to (assignemnt is random).

        self.con_n_SST_to_PN = 1 				# Number of SST neurons that are connected to each PN at random.
        self.con_n_VIP_to_SST = 1 				# Number of VIP neurons that are connected to each PN at random.

        self.p_PN = 0.4
        self.p_PV = 0.42 						# Probability of a connection from a PN to an PV neuron (meaning that one PN is connected to ca. 30 % of all PVs)
        self.p_SST = 0.19						# Probability of a connection from a PN to an SST neuron (meaning that one PN is connected to ca. 19 % of all SSTs)
        # Population weights
        self.pop_weight_Input_PN = 2.7
        self.pop_weight_PN_PN = 0.07       # These weights are trainable
        self.pop_weight_PN_PV = 0.05
        self.pop_weight_PV_PN = 0.015
        self.pop_weight_PN_SST = 0.1
        self.pop_weight_SST_PN = 0.15
        self.pop_weight_VIP_SST = 4


        # Decay constant for EPSP
        self.k_PN = 0.0526                                                  # Decay constant of PN membrane potential V_PN.
        self.k_SST= 0.1608                                                  # Decay constant of the SST neuron membrane potential.
        self.k_VIP = 0.184                                                  # Decay constant of the VIP neuron membrane potential.
        self.k_PV = 0.25                                                    # Decay constant of the VIP neuron membrane potential.

        # Parameters for activation function
        # Maximal firing rates
        self.Rmax_PN = 150
        self.Rmax_PV = 193
        self.Rmax_SST = 199
        self.Rmax_VIP = 196

        # Resting Membrane potential for all neurons
        self.Vrest_PN = 0
        self.Vrest_PV = 0
        self.Vrest_SST = 0
        self.Vrest_VIP = 0

        # Thresholds for depolarization and firing of the neurons in units of membrane potential.
        self.Thresh_PN = 5
        self.Thresh_PV = 30
        self.Thresh_SST = 15
        self.Thresh_VIP = 5                       # Set arbitrarly

        # Gains for ReLU function
        self.Gain_PN = 1
        self.Gain_PV = 2.7
        self.Gain_SST = 1.6
        self.Gain_VIP = 2.2                       # Mean of SST and PV (no literature)


        ### Parameters for learning ###

        # Set parameters for OJA learning
        self.alpha = 0.8 	                                             # Rate of regularization in OJA.
        self.epsilon = 0                                                 # Threshold for weight update.
        self.timeAvrg = 100                                              # Time over which is integrated in a learning step (Ojas rule)

        # Track outer products
        self.outr_act = np.zeros((self.timeAvrg, self.n_PN, self.n_PN))  # Stores the outer products of the activites. Used for integration in Ojas rule.
        ###############################################################

        ### Set time of odor presentation in milliseconds & gap between odor presentation
        self.n_burst = 100 												# Number of bursts once the input fires // time the input is ON in ms.
        self.empty_burts = 2 											# Gap in "n_bursts" between two odors (here 200 ms).
        self.scale = 0.004
        ###############################################################


        ### Initialize Membrane potentials and activities of all the neurons over time ###

        # Track the membrane potentials of neurons
        self.V_PN = np.zeros([self.n_PN,self.time_ms])
        self.V_PV = np.zeros([self.n_PV,self.time_ms])
        self.V_SST = np.zeros([self.n_SST,self.time_ms])
        self.V_VIP = np.zeros([self.n_VIP,self.time_ms])

        # Initialize with resting membrane potential
        self.V_PN[:,0] = self.Vrest_PN
        self.V_PV[:,0] = self.Vrest_PV
        self.V_SST[:,0] = self.Vrest_SST
        self.V_VIP[:,0] = self.Vrest_VIP

        # Track the activities of the neurons.
        self.Activity_PN = np.zeros([self.n_PN,self.time_ms])
        self.Activity_PV = np.zeros([self.n_PV,self.time_ms])
        self.Activity_SST = np.zeros([self.n_SST,self.time_ms])
        self.Activity_VIP = np.zeros([self.n_VIP,self.time_ms])

        ###############################################################

        ### Initialize vectors to store input and teacher signal

        self.teacher = np.zeros((self.n_VIP, self.time_ms))					# Stores the teacher signal over whole training time
        self.input = np.zeros((self.n_inputs,self.time_ms))					# Input array for each timestep
        self.RepInptHist = np.zeros((self.n_randInputs, self.time_ms))      # Stores the position of the input that is presented repeatedly

        ###############################################################
        self._InputandTeacher()
        self._setConnectionsAndWeights()
        ### Used in plotting functions
        self.time_s = int(time_ms / 1000)

    def _setConnectionsAndWeights(self):
        '''
        Function initializes connection matrices for all the neurons using the probabilities defined in __init__.
        '''

        ### CONNECTIONS FOR UNTRAINABLE WEIGHTS
            # The connectivity matrix directly includes the constant weights.

        # SEED
        rng = np.random.default_rng(self.seed)

        # INPUT
        # Set connections and strength from input stream to PN neurons. Here the weights are included into the connection matrix.
        A = np.zeros((self.n_PN,self.n_inputs))
        A[:,:self.numberOfGlomeruliToPN] = 1
        self.weightInput_PN = rng.permuted(A, axis=1) * np.random.uniform(0, 1, size=(self.n_PN, self.n_inputs))
        n_weights = self.n_PN*self.numberOfGlomeruliToPN
        Inpt_PN_factor = np.sum(self.weightInput_PN) / n_weights
        self.weightInput_PN = (self.pop_weight_Input_PN / Inpt_PN_factor) * self.weightInput_PN

        # PN TO PV
        # Set connecntions from PN to PV
        self.weightPN_PV = np.random.choice([0, 1], size=(self.n_PV, self.n_PN), p=[1-self.p_PV, self.p_PV])
        self.weightPN_PV = self.weightPN_PV * self.pop_weight_PN_PV     			# Since the average weight is equal to 1, does only work as long as all weights the same!

        # PN TO SST
        # Set connections from PN to SST
        self.weightPN_SST =  np.random.choice([0, 1], size=(self.n_SST, self.n_PN), p=[1-self.p_SST, self.p_SST])
        self.weightPN_SST = self.weightPN_SST * self.pop_weight_PN_SST 			# Since the average weight is equal to 1, does only work as long as all weights the same!

        # VIP TO SST
        # Set connections and strength from VIP to SST neurons. Here the weights are included into the connection matrix.
        A = np.zeros((self.n_SST,self.n_VIP))
        A[:,:self.con_n_VIP_to_SST] = 1
        self.weightVIP_SST= rng.permuted(A, axis=1)
        VIP_SST_factor = np.sum(self.weightVIP_SST)
        self.weightVIP_SST = self.weightVIP_SST * (self.pop_weight_VIP_SST / VIP_SST_factor)


        ### CONNECTIONS FOR TRAINABLE WEIGHTS
        # The connectivity matrix does not include weights.

        # RECURRENT PN TO PN CONNECTIONS

        # PN TO PN
        # Set recurrent connections from PN to PN
        self.conPN_PN = np.random.choice([0, 1], size=(self.n_PN, self.n_PN), p=[1-self.p_PN, self.p_PN])
        np.fill_diagonal(self.conPN_PN, 0)  # Set weights for recurrent connections back to the same neuron to zero (Not possible to train using Ojas rule)

        # Allocate memory for weight matrix and randomly initialize weights
        self.weightPN_PN = np.zeros((self.time_ms,self.n_PN, self.n_PN))
        self.weightPN_PN[0,:,:] = np.random.uniform(0, 1, size=(self.n_PN, self.n_PN)) * self.conPN_PN
        n_weights = np.sum(self.conPN_PN)
        PN_PN_factor = self.pop_weight_PN_PN / (np.sum(self.weightPN_PN[0,:,:]) / n_weights)
        self.weightPN_PN[0,:,:] = self.weightPN_PN[0,:,:] * PN_PN_factor        # Weights are normalized such that the population average strength, independent of the number of connecitons, is 7 (Paper: Differential Excitability of PV and SST Neurons Results in...

        # SST TO PN
        # Set connections from SST to PN
        A = np.zeros((self.n_PN,self.n_SST))
        A[:,:self.con_n_SST_to_PN] = 1
        self.weightSST_PN = rng.permuted(A, axis=1)

        # PV TO PN
        # Set connections from PV to PN
        A = np.zeros((self.n_PN,self.n_PV))
        A[:,:self.con_n_PV_to_PN] = 1
        self.weightPV_PN = rng.permuted(A, axis=1)

        # SST to PN
        self.weightSST_PN = self.weightSST_PN * self.pop_weight_SST_PN
        # PV to PN
        self.weightPV_PN = self.weightPV_PN * self.pop_weight_PV_PN


    def _InputandTeacher(self):
        '''
        Creates the input and teaching signal.

        '''

        if (int(self.time_ms) % int(self.n_burst)) != 0:
            print("Simulation time needs to be divisible by burst size")
            exit()

        if (self.n_inputs < self.numberOfActiveInputs):
            print("The size of the input vector needs to be larger than the number of active inputs.")
            exit()

        # Create two distinct input vectors.
        x1,x2= np.zeros(self.n_inputs),np.zeros(self.n_inputs)
        x1[0:self.numberOfActiveInputs] = 1
        x2[self.numberOfActiveInputs:2*self.numberOfActiveInputs] = 1

        # Variable used to alternate between the three different inputs (x1, x2 and rand(x2))
        alternate = 1
        for i in range(self.time_ms):

            # Generate the input with teaching signal
            if i % ((self.empty_burts + 1)*self.n_burst) == 0 and alternate == 1:
                for j in range(self.n_burst):
                    self.input[:,i+j] = x2
                    self.teacher[0,i+j] = 10
                alternate = 2
                continue

            # Generate a random input to present between the two recurring once.
            if i % ((self.empty_burts + 1)*self.n_burst) == 0 and alternate == 2:
                rand = np.random.permutation(x2)
                for j in range(self.n_burst):
                    self.input[:,i+j] = rand
                alternate = 3
                continue

            # Generate the periodic random input
            if i % ((self.empty_burts + 1)*self.n_burst) == 0 and alternate == 3:
                for j in range(self.n_burst):
                    self.input[:,i+j] = x1
                    self.RepInptHist[0,i+j] = 1
                alternate = 1
                continue

    def getEnsembles(self):
        '''
        The function extracts neuronal ensembles for each odor presented repeatedly.
        '''
        # Neurons that show activity at the very last input presentation and are not decreasing in activity are counted as ensembles.

        min_activity = 10
        #max_negativeSlope = -1
        #timewindow_Slope = 15

        self.ensembleNeurons = []
        self.ensembleActivites = []
        self.ensembleActivitesUntrained = []
        self.ensembleNeuronsUntrained = []

        # Get ensemble neurons and activities where teacher signal is present.
        for i in range(self.n_VIP):
            # Get indices (of timepoint) of untraind (idx0) and trained (idx).
            idx0 = np.where(self.teacher[i,:].astype(bool) == True)[0][self.n_burst-1] # Get index of first presentation of corresponding VIP input (index of last input presentation in first burst)
            idx = np.where(self.teacher[i,:].astype(bool) == True)[0][-1] # Get index of last presentation of corresponding VIP input


            # Get Neuron numbers and activities after training
            self.ensembleNeurons.append(np.where(self.Activity_PN[:,idx] > min_activity)[0].tolist()) # Append indices (neuron number) for each ensemble. & ((self.Activity_PN[:,idx] - self.Activity_PN[:,idx-1]) > 0) #& (np.average(np.diff(self.Activity_PN[:,idx-timewindow_Slope:idx]),axis = 1) > max_negativeSlope)
            self.ensembleActivites.append(self.Activity_PN[self.ensembleNeurons[-1],idx].tolist()) # Append activities of the neurons in ensemble.

            # Get Neuron numbers and activities before training
            self.ensembleActivitesUntrained.append(self.Activity_PN[self.ensembleNeurons[-1],idx0].tolist()) # Append activities of the untrained neurons in ensemble.


        for j in range(self.n_randInputs):
            # Get ensemble neurons and activities for repeatly presented input w/o teacher signal. It is stored as the last list.
            idx0 = np.where(self.RepInptHist[j,:].astype(bool) == True)[0][self.n_burst-1]
            idx = np.where(self.RepInptHist[j,:].astype(bool) == True)[0][-1] # Get index of last presentation of corresponding VIP input
            self.ensembleNeurons.append(np.where(self.Activity_PN[:,idx] > min_activity)[0].tolist()) # Append indices (neuron number) for each ensemble. #& ((self.Activity_PN[:,idx] - self.Activity_PN[:,idx-1]) > 0)
            self.ensembleActivites.append(self.Activity_PN[self.ensembleNeurons[-1],idx].tolist()) # Append activities of the neurons in ensemble.

            self.ensembleActivitesUntrained.append(self.Activity_PN[self.ensembleNeurons[-1],idx0].tolist()) # Append activities of the untrained neurons in ensemble.

    # These are plotting functions that are assigned to the class from another file (plottingFunctions.py).
    plotActivityPub = plotActivityPub
    plotNetwork = plotNetwork
    plotConnectivity = plotConnectivity
    plotEnsembleWeights = plotEnsembleWeights
    plotEnsembleStrength = plotEnsembleStrength
    plotWeightDistribution = plotWeightDistribution
    plotInput = plotInput

Now, that we have the basis of our model, we need to actually simulate the equations that describe the neurons. Let's do that in the `feedForward()` function below. Fill in the missing lines of code below for the voltage potential of the PV, SST and VIP neuron, which are indicated with `...`

In [2]:
def feedForward(self,i):
    
    ### Dimensions of variables:
    # self.weightInput_PN: (n_PN x n_inputs)
    # self.weightPN_PN: (time,n_PN,n_PN)
    # self.weightPV_PN: (n_PN,n_PV)
    # self.weightPN_PV: (n_PV,n_PN)
    # self.weightPN_SST: (n_SST,n_PN)
    # self.weightVIP_SST: (n_SST,n_VIP)
    # self.input: (n_inputs, time_ms)
    # self.V_X: (n_X, time_ms), X = {PN, PV, SST, VIP}
    # self.Activity_X: (n_X, time_ms), X = {PN, PV, SST, VIP}

    ### Other variables 
    # i indexes the time.
    # self.teacher: (n_VIP, time_ms)
    # self.k_X: (single value) 

    ### Simulate membrane potential of PN, SST, PV & VIP neurons. ###

    # Calculate the activation by the input as well as the recurrent activity (rec_act) to use below.
    Input =  (self.weightInput_PN @ self.input[:,i])
    rec_act = (self.scale*self.weightPN_PN[i-1,:,:] @ self.Activity_PN[:,i-1])

    # PN: V_PN(t-1) + Input + recurrent connection - self decay - SST inhibition (max equal to odor input (distal)) - PV inhibition
    self.V_PN[:,i] = self.V_PN[:,i-1] + Input + rec_act - (self.k_PN * self.V_PN[:,i-1])  - np.minimum(Input, self.weightSST_PN @ self.Activity_SST[:,i-1]) - (self.weightPV_PN @ self.Activity_PV[:,i-1])

    # PV:
    self.V_PV[:,i] = ...

    # SST: 
    self.V_SST[:,i] = ...

    # VIP: 
    self.V_VIP[:,i] = ...

    # Convert to activity using the capped ReLU function
    self.Activity_PN[:,i] = np.maximum(0,np.minimum(self.Gain_PN*(self.V_PN[:,i]-self.Thresh_PN),self.Rmax_PN))
    self.Activity_PV[:,i] = np.maximum(0,np.minimum(self.Gain_PV*(self.V_PV[:,i]-self.Thresh_PV),self.Rmax_PV))
    self.Activity_SST[:,i] = np.maximum(0,np.minimum(self.Gain_SST*(self.V_SST[:,i]-self.Thresh_SST),self.Rmax_SST))
    self.Activity_VIP[:,i] = np.maximum(0,np.minimum(self.Gain_VIP*(self.V_VIP[:,i]-self.Thresh_VIP),self.Rmax_VIP))

# This line adds the function to the APC_circuit class
APC_circuit.feedForward = feedForward # Add function to APC_circuit class

The last thing we need to do is to write a function that does the actual simulation. This is implemented in the `simulate()` function below.

In [3]:
def simulate(self):
    # dt = 1ms
    # number of ms to simulate is stored in variable: self.time_ms

    st = time.time() # Timestamp at start of simulation

    # Actual simulation
    for i in range(1,self.time_ms):
        self.feedForward(i)

    et = time.time() # Timestamp at end of simulation
    print('Simulation complete after', et - st, 'seconds') # Print time duration of simulation

# This line adds the function to the APC_circuit class
APC_circuit.simulate = simulate # Add function to APC_circuit class

Now that we have the whole network set up, let's create an instance of the class and simulate odor input to the network for 40 seconds, which is the default value specified in the `__INIT__()` function.

In [ ]:
mynn = APC_circuit()
mynn.simulate()

We have now simulated and saved all the neurons' activities. Let's first check the recurrent connectivity between the PNs:

You can check if your output is correct by comparing it with the graphs in the file `Expected Outputs.pdf`.

In [ ]:
mynn.plotConnectivity() # Output 1

As you can see, we have a recurrent connectivity between PNs of about 40 % (as specified in `__INIT__()`) and no PN projects back to itself (empty diagonal in connectivity matrix). Let's have a look at the activity of the different neurons at the beginning and at the end of the simulation. Each color corresponds to the activity of a single neuron.

In [ ]:
mynn.plotActivityPub() # Output 2

Next, we extract the neuronal ensembles that respond to each of the two odors and check their activity before and after training:

In [ ]:
mynn.getEnsembles() # This function extracts the neuronal ensembles for the teaching signal linked odor as well as for the control odor.
mynn.plotEnsembleStrength() # Output 3

As you can see, and probably expected, the activity profile at the beginning and end of the simulation is identical. This is because we haven't implemented any learning at all - let's change that now.
We will only implement learning for the recurrent PN-to-PN connections and use one of the simplest unsupervised learning rules: Hebbian learning.


$$\frac{\Delta w}{\Delta t} = \lambda \cdot x \cdot y $$
$$ w(t)  = w(t-1) +  \Delta w(t) $$

Where $x = r(t-1)$ is the presynaptic activation, $y = r(t)$ is the postsynaptic activation, $w$ is the weight between the neurons and $\lambda$ is the learning rate. <br>

The equation above gives the learning rule for a single pre- and postsynaptic neuron. What would the rule look like for multiple PNs, i.e. in vector format (in which case $W$ is a matrix)?
The weight matrices are of the form:

$$ \begin{bmatrix}
w_{1,1} & w_{1,2}\\
w_{2,1} & w_{2,2}
\end{bmatrix}  $$

Where the weight $w_{1,2}$ is the weight of the connection from Neuron 2 $\to$ Neuron 1.

Your answer:

$$\frac{\Delta W}{\Delta t} = ... $$

Below is the implementation. The weight between the PNs is `self.weightPN_PN[x,y,z]`, where `x` indexes the time and `y` and `z` index the row and column of the weight matrix. Fill in the missing line of code below for the $\Delta W$ of the PN-to-PN connections, indicated by `...`.

In [8]:
def hebbian(self,i):

    d_weight = ...
    self.weightPN_PN[i,:,:] = np.maximum((d_weight * self.conPN_PN) + self.weightPN_PN[i-1,:,:],0)      # Only update weights that really exist and only if they stay positive.

APC_circuit.hebbian = hebbian

Next, add the missing line of code to include the weight update in the `simulate()` function, which is indicated by `...`

In [9]:
def simulate(self):
    # dt = 1ms
    # number of ms to simulate is stored in variable: self.time_ms

    st = time.time() # Timestamp at start of simulation

    # Actual simulation
    for i in range(1,self.time_ms):
        self.feedForward(i)
        ...

    et = time.time()# Timestamp at end of simulation
    print('Simulation complete after', et - st, 'seconds') # Print time duration of simulation

APC_circuit.simulate = simulate # Add function to APC_circuit class

Let's run the simulation again and check, if the activities changed this time.

In [ ]:
# Output 4
mynn = APC_circuit()
mynn.simulate()
mynn.plotActivityPub()
mynn.getEnsembles() # This function extracts the neuronal ensembles for the teaching signal-linked odor as well as for the control odor.
mynn.plotEnsembleStrength()

It looks like all inhibitory neurons and all PN's that were recruited into an odor encoding neuronal ensemble fire at their maximum firing rate. Next, let's look at the weight distribution.

In [ ]:
mynn.plotWeightDistribution() # Output 5

As you can see, some of the weights, and therefore some of the activities, explode in Hebbian learning. As you know, neurons have limited resources and the weights between synapses (e.g. the number of AMPA receptors) cannot increase indefinitely. Furthermore, as you learned in the second lecture, neurons tend to normalise their activity to a given baseline. A Hebbian-based learning rule that incorporates this concept is Oja's learning rule:
<a id='oja'></a>
$$\Delta W(t) = \frac{\lambda}{T}\left[\int_{t - T}^{t} \vec{r}(u) \vec{r}^\top(u-1) du - W \alpha \int_{t - T}^{t} \overline{r}(u)^2 du\right]$$

Weightupdate:
$$ W(t)  = W(t-1) +  \Delta W(t) $$

To make the learning rule smoother, we average over a time window specified in the variable `self.timeAvrg` which is set to $100~ms$. Since we are modelling time in discrete steps, we are actually summing and not integrating.


In [12]:
def _Oja(self, i):
    j = min(i,self.timeAvrg)                                                                            # Stores the indicies for the Integration

    self.outr_act[i%self.timeAvrg,:,:] = np.outer(self.Activity_PN[:,i],self.Activity_PN[:,i-1])        # Stores outer products in a circular buffer fashion, over the self.timeAvrg time points (Hebbian part of Oja rule)

    # Calculate integral
    int_1 = (1/j) * np.sum(self.outr_act,axis=0)                                                        # Calculates the sum over "timeAvrg" for the input*output matrix. Axis = 0 makes sure to sum over the time dimension.

    avr_output_sq = np.average(self.Activity_PN[:,i-j:i],axis=1)**2                                     # Calculate the normalizing part of Ojas rule (squared average over time (TIME OJA))

    int_2 =  avr_output_sq[:,np.newaxis] * self.weightPN_PN[i-1,:,:]

    d_weight = self.learning_rate   * (int_1 -  self.alpha * int_2)                                     # Finally, calculate the weight update.
    self.weightPN_PN[i,:,:] = np.maximum((d_weight * self.conPN_PN) + self.weightPN_PN[i-1,:,:],0)      # Only update weights that really exist and only if they stay be positive.

APC_circuit._Oja = _Oja # Add function to APC_circuit class

We have to update the `simulate()`function again:

In [13]:
def simulate(self):
    # dt = 1ms
    # number of ms to simulate is stored in variable: self.time_ms
    st = time.time() # Timestamp at start of simulation

    # Actual simulation
    for i in range(1,self.time_ms):
        self.feedForward(i)
        self._Oja(i)


    et = time.time() # Timestamp at end of simulation
    print('Simulation complete after', et - st, 'seconds') # Print time duration of simulation

APC_circuit.simulate = simulate # Add function to APC_circuit class

Let's go through the whole process one last time and see how Oja's rule changes the network's learning behaviour.

In [ ]:
# Output 6
mynn = APC_circuit(time_ms = 100000)
mynn.simulate()
mynn.getEnsembles()
mynn.plotActivityPub() # Graph 6
mynn.plotEnsembleStrength() 
mynn.plotWeightDistribution()

This looks much more realistic. Now let's have a look at our trained network. The `plotNetwork()` function plots each PN as a vertex and the sum of weights between neurons as edges. Strongly connected neurons are closer together than weakly connected neurons.

In [ ]:
# Output 7
mynn.plotNetwork() # Ignore the RuntimeWarning.

Finally, let's have a look at the evolution of the average within-ensemble weights. As you can already see in the figure above, we expect them to increase much more over time than the weights between neurons that are not part of the neuronal ensemble (which we call "between" in the figure below).

In [ ]:
mynn.plotEnsembleWeights() # Output 8

As you can see, activating the disinhibitory circuit during odor presentation leads to a faster learning of the odor, as well as to larger neuronal ensembles in general.


**Questions**
1. Sometimes the control-odor is learned faster than the teaching-odor, depending on the network initialization. Why could this be possible? (Hint: Neuronal ensembles formed for the control odor often consist of fewer neurons).
2. What would the weight distribution look like if we normalised not by the L2 norm, as in Oja's rule, but by the L1 norm?
4. What is the effect of $\alpha$ in [Oja's rule](#oja)? What happens if $\alpha$ is too large or too small?

**Your Answers:**
1. ...
2. ...
3. ...

# References
<a id='ref'></a>

Canto-Bustos, M., F. Kathryn Friason, C. Bassi, and A. M. M. Oswald: Disinhibitory Circuitry Gates Associative Synaptic Plasticity in Olfactory Cortex. The Journal of neuroscience: the official journal of the Society for Neuroscience, 42(14):2942–2950, 2022. doi: 10.1523/JNEUROSCI.1369-21.2021.

Shepherd, G. M.: Computational structure of the olfactory system. Olfaction: A model system for computational neuroscience, pp. 3–42, 1991.

Uchida, N., C. Poo, and R. Haddad: Coding and transformations in the olfactory system, 2014.